# Capítulo 8 — Predição de produtividade com Machine Learning

**Curso:** Agrometeorologia Operacional com Python
**Prof. Dr. Fabrício Correia de Oliveira** — UTFPR, Campus Santa Helena

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologia/blob/main/curso/08_machine_learning.ipynb)

> Pré-requisito: Capítulos 1 a 7 (em especial o ISNA do Capítulo 7).

---


## 8.1 Motivação

O ISNA do Capítulo 7 responde uma pergunta binária/ordinal: essa janela de semeadura foi
melhor ou pior que aquela? Mas não diz **quanto** de produtividade esperar. Para isso,
precisamos de um modelo estatístico que aprenda, a partir de exemplos históricos, a relação
entre variáveis agroclimáticas (ISNA, temperatura, déficit hídrico...) e produtividade —
esse é o papel do Machine Learning neste curso, na mesma linha do pipeline usado no artigo
de milho segunda safra (balanço hídrico agroclimático → ML → priorização de risco).

Vamos usar o **Extra Trees Regressor** (Extremely Randomized Trees): um ensemble de árvores
de decisão que costuma performar bem em problemas agronômicos com poucas amostras e relações
não lineares entre variáveis, sem exigir muito ajuste de hiperparâmetros.


## 8.2 Do problema ao código

Bibliotecas novas neste capítulo: `scikit-learn`, o padrão de Machine Learning em Python.
O fluxo é sempre o mesmo, independente do modelo: **separar treino/teste → treinar → avaliar
→ interpretar**.


In [ ]:
%pip install -q scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score


## 8.3 Atividade guiada — dataset didático

> **Atenção:** assim como no Capítulo 7, os dados abaixo são **sintéticos** — a produtividade
> foi gerada por uma fórmula conhecida (mais ruído aleatório), de propósito. Isso permite
> verificar se o modelo consegue "redescobrir" a relação real, algo impossível de checar com
> dados reais, cuja relação verdadeira ninguém conhece de antemão.
>
> Em um projeto real, a coluna de produtividade viria de dados observados — por exemplo, do
> Levantamento Sistemático da Produção Agrícola (LSPA) ou do Censo Agropecuário/PAM do IBGE,
> por município e safra.


In [ ]:
rng = np.random.default_rng(7)
n_amostras = 120  # ex.: vários municípios x várias combinações de ano/janela de semeadura

ISNA = rng.uniform(0.5, 1.0, n_amostras)
Ta = rng.uniform(18, 25, n_amostras)     # temperatura média anual (°C)
Dha = rng.uniform(0, 200, n_amostras)    # deficiência hídrica anual (mm) — Capítulo 6/7

# relação conhecida usada para gerar os dados (o modelo NÃO tem acesso a isto):
produtividade = (
    2000
    + 6000 * ISNA
    + 150 * (Ta - 20)
    - 5 * Dha
    + rng.normal(0, 300, n_amostras)
)
produtividade = np.clip(produtividade, 1000, None)

df_treino = pd.DataFrame({
    "ISNA": ISNA, "Ta": Ta, "Dha": Dha, "produtividade_kg_ha": produtividade
})
df_treino.describe().round(2)


## 8.4 Treino e avaliação do modelo


In [ ]:
X = df_treino[["ISNA", "Ta", "Dha"]]
y = df_treino["produtividade_kg_ha"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

modelo = ExtraTreesRegressor(n_estimators=300, random_state=42)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"RMSE (teste) = {rmse:.1f} kg/ha")
print(f"R²   (teste) = {r2:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, y_pred, alpha=0.7)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lim, lim, "--", color="gray", label="predição perfeita")
ax.set_xlabel("Produtividade observada (kg/ha)")
ax.set_ylabel("Produtividade prevista (kg/ha)")
ax.set_title("Extra Trees — observado x previsto (conjunto de teste)")
ax.legend()
plt.tight_layout()
plt.show()


## 8.5 Interpretando o modelo — importância das variáveis

Diferente de um modelo de regressão linear, o Extra Trees não dá um "coeficiente" por
variável — mas dá a **importância relativa** de cada uma nas decisões das árvores. É uma
forma de checar se o modelo aprendeu algo agronomicamente plausível.


In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(6, 3))
importancias.plot(kind="barh", ax=ax, color="seagreen")
ax.set_xlabel("Importância relativa")
ax.set_title("Importância das variáveis — Extra Trees")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

importancias.round(3)


Nos dados sintéticos deste capítulo, `ISNA` foi construído como a variável de maior peso na
fórmula geradora — se o gráfico acima também aponta `ISNA` como a mais importante, é um bom
sinal de que o modelo está capturando a relação real, e não ruído.


## 8.6 Por que validação cruzada importa

Uma única divisão treino/teste pode, por acaso, cair numa combinação fácil (ou difícil) de
prever. **Validação cruzada (k-fold)** treina e avalia o modelo várias vezes, em partições
diferentes dos dados, e reporta a variação do desempenho — essencial quando o dataset é
pequeno, como costuma ser o caso em agroclimatologia municipal (dezenas de municípios, não
milhões de registros).


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores_r2 = cross_val_score(modelo, X, y, cv=cv, scoring="r2")

print("R² em cada uma das 5 partições:", scores_r2.round(3))
print(f"R² médio: {scores_r2.mean():.3f}  (desvio-padrão: {scores_r2.std():.3f})")


## 8.7 Desafio

1. Reduza `n_amostras` para 30 (simulando um cenário realista de poucos municípios/anos) e
   repita o treino e a validação cruzada. O que acontece com o `R²` médio e com seu
   desvio-padrão entre as partições? O que isso te diz sobre a confiabilidade de um modelo
   treinado com poucos dados?
2. Adicione uma quarta variável ao dataset sintético (por exemplo, altitude do município,
   `rng.uniform(300, 1200, n_amostras)`) **sem** incluí-la na fórmula que gera a produtividade.
   Treine o modelo novamente — a importância dessa variável deveria ficar baixa. Ela ficou?
3. *(projeto real)* Substitua o dataset sintético pelas variáveis reais calculadas nos
   Capítulos 6 e 7 (ISNA, Ta, Dha) para os municípios que você trabalhou, e por produtividade
   observada (IBGE/LSPA) — essa é a versão de produção deste pipeline.


In [ ]:
# Espaço para o desafio — escreva seu código aqui


## 8.8 Checkpoint — fechando o pipeline do curso

Você percorreu, capítulo a capítulo, o caminho completo de um projeto de agrometeorologia
operacional:

`dado bruto (Cap. 1)` → `radiação (Cap. 2)` → `graus-dia/fenologia (Cap. 3)` →
`umidade/energia (Cap. 4)` → `evapotranspiração (Cap. 5)` → `balanço hídrico (Cap. 6)` →
`ISNA/zoneamento (Cap. 7)` → `predição de produtividade (Cap. 8)`.

Cada capítulo produziu uma função testada e validada contra um exercício com resultado
conhecido, e cada capítulo reaproveitou as funções dos anteriores — exatamente como um
pipeline de produção real é construído.

Os Encontros 9 e 10 são de avaliação (formato a definir pelo professor), cobrindo este
pipeline completo, idealmente aplicado a dados reais de uma cultura e região à escolha.
